# FCM Code

### Import Modules

In [15]:
import mlhp
import datetime
import os
import numpy as np
import matplotlib.pyplot as plt
from vtk import vtkPolyData, vtkPoints, vtkCellArray, vtkLine, vtkXMLPolyDataWriter, vtkVertex
from math import sqrt

### Track start time & define workspaces

In [16]:
starttime=datetime.datetime.now()
outputDir="/Applications/Datein_Timon/Bildung/Uni_Rostock/Master_MaschBau/3_Semester/Studienarbeit/Code/FCM-StA/OutputsSolidHomogenStress" 

### 1. Setting up mesh and basis

In [ ]:
D = 3

# Setup triangulation domain
triangulation = mlhp.readStl("/Applications/Datein_Timon/Bildung/Uni_Rostock/Master_MaschBau/3_Semester/Studienarbeit/Code/Inputs/20mm_cube.stl")
#triangulation = mlhp.readStl("/Applications/Datein_Timon/Bildung/Uni_Rostock/Master_MaschBau/3_Semester/Studienarbeit/Code/FCM-StA/FCM/Wuerfel1.stl")

##würfel faktor 10 zu groß
scale = 0.006/20  # kantenlänge von 0.006m
scale =1
# Umwandeln in NumPy-Array
verts = np.array(triangulation.vertices)
# Skalieren
verts_scaled = verts * scale
# Zurückschreiben in triangulation
triangulation.vertices = verts_scaled

kdtree = mlhp.buildKdTree(triangulation)
domain = mlhp.implicitTriangulation(triangulation, kdtree)

# Setup discretization
youngsModulus = 1e10 # youngs modulus in Pa (=10GPa)
poissonsRatio = 0.3

polynomialDegree = 1 
nelements = [1]*D # original value was 50
alphaFCM = 1e-5
penalty = 1e5 * youngsModulus

origin, max = triangulation.boundingBox()
lengths = [m - o for o, m in zip(origin, max)] 
origin = [o - 1e-20 for o in origin]
max = [m + 1e-20 for m in max]
#debugging
print("STL Bounding Box:")
print("  Origin :", origin)
print("  Max    :", max)
print("  Length :", lengths)
print("  Fix unten bei z <", origin[2] + lengths[2]*0.01)
print("  Drück oben bei z >", origin[2] + lengths[2]*0.99)

#grid = mlhp.makeRefinedGrid(nelements, lengths, origin)# wenn DOF von 1640442 auf 3090903 gehen alles klar
grid = mlhp.makeGrid(nelements, lengths, origin) #grobes gitter
grid = mlhp.makeRefinedGrid(mlhp.makeFilteredGrid(grid, domain=domain, nseedpoints=polynomialDegree + 2))#filterung mit stl-domain

basis = mlhp.makeHpTensorSpace(grid, polynomialDegree, nfields=D)
print(basis)

STL Bounding Box:
  Origin : [-1e-20, -20.0, -1e-20]
  Max    : [20.0, 1e-20, 20.0]
  Length : [20.0, 20.0, 20.0]
  Fix unten bei z < 0.2
  Drück oben bei z > 19.8
MultilevelHpBasis<3> (address: 0x318f24618)
    number of elements         : 1
    highest polynomial degree  : 1
    number of unknowns         : 24
    number of field components : 3
    average dofs per element   : 24.0
    heap memory usage          : 296.0 bytes



#### 2. Probe line generation and export to ParaView

In [18]:
# Mitte von Y und Z
mid_y = origin[1] + 0.5 * lengths[1]
mid_z = origin[2] + 0.5 * lengths[2]

# 50 Punkte entlang X zur Visualisierung
x_vals = np.linspace(origin[0], max[0], 50)
line_pts = [(x, mid_y, mid_z) for x in x_vals]

# Punkte, die wirklich im Körper liegen (optional: kann später mit domain geprüft werden)
inside_line_pts = [(i, pt) for i, pt in enumerate(line_pts) if domain(pt) > 0]

# 10 gleichmäßig verteilte Punkte
if len(inside_line_pts) >= 10:
    step = len(inside_line_pts) // 9
    selected_pts = inside_line_pts[::step][:10]
else:
    selected_pts = inside_line_pts

# Erzeuge VTK-Punkte
vtkpoints = vtkPoints()
for pt in line_pts:
    vtkpoints.InsertNextPoint(pt)

# Erzeuge VTK-Linie
lines = vtkCellArray()
for i in range(len(line_pts) - 1):
    line = vtkLine()
    line.GetPointIds().SetId(0, i)
    line.GetPointIds().SetId(1, i + 1)
    lines.InsertNextCell(line)

# Erzeuge VTK-Vertices für die ausgewählten Punkte
vertices = vtkCellArray()
for idx, pt in selected_pts:
    vertex = vtkVertex()
    vertex.GetPointIds().SetId(0, idx)  # ✅ direkter Index
    vertices.InsertNextCell(vertex)

# Kombiniere alles zu einem PolyData-Objekt
polydata = vtkPolyData()
polydata.SetPoints(vtkpoints)
polydata.SetLines(lines)
polydata.SetVerts(vertices)

# Exportiere
writer = vtkXMLPolyDataWriter()
writer.SetFileName(outputDir + "/cutline_with_points.vtp")
writer.SetInputData(polydata)
writer.Write()

print("Exported cutline and 10 points to 'cutline_with_points.vtp'")

Exported cutline and 10 points to 'cutline_with_points.vtp'


### 3. Allocating linear system

In [19]:
matrix = mlhp.allocateSparseMatrix(basis)
vector = mlhp.allocateRhsVector(matrix)

### 4. Computing weak boundary integrals

In [20]:
def createBoundaryQuadrature(func):
     filtered = mlhp.filterTriangulation(triangulation, mlhp.implicitFunction(D, func))
     intersected, celldata = mlhp.intersectTriangulationWithMesh(grid,filtered)
     quadrature = mlhp.triangulationQuadrature(intersected, celldata,polynomialDegree + 1)
     return intersected, celldata, quadrature

intersected0, celldata0, quadrature0 = createBoundaryQuadrature(f"z <{origin[2] + 0.01*lengths[2]}")#oben
intersected1, celldata1, quadrature1 = createBoundaryQuadrature(f"z >{origin[2] + 0.99*lengths[2] }")#unten
intersected2, celldata2, quadrature2 = createBoundaryQuadrature(f"x <{origin[0] + 0.01*lengths[0]}")#links
intersected3, celldata3, quadrature3 = createBoundaryQuadrature(f"x >{origin[0] + 0.99*lengths[0] }")#rechts

strain=0.1
integrand0 = mlhp.l2BoundaryIntegrand(mlhp.vectorField(D, [0,0,penalty]), mlhp.vectorField(D, [0.0, 0.0, 0.0]))#wird in z richtung festgehalten & in z bestraft
integrand1 = mlhp.l2BoundaryIntegrand(mlhp.vectorField(D, [0,0,penalty]), mlhp.vectorField(D, [0, 0.0, strain*lengths[2]*penalty])) # verschiebung in z im 1%
integrand2 = mlhp.l2BoundaryIntegrand(mlhp.vectorField(D, [penalty,0,0]), mlhp.vectorField(D, [0.0, 0.0, 0.0]))#nur in x richtung festhalten
integrand3 = mlhp.l2BoundaryIntegrand(mlhp.vectorField(D, [penalty,0,0]), mlhp.vectorField(D, [0, 0.0, 0.0])) #nur in x richtung festhalten

mlhp.integrateOnSurface(basis, integrand0, [matrix, vector], quadrature0)
mlhp.integrateOnSurface(basis, integrand1, [matrix, vector], quadrature1)
#mlhp.integrateOnSurface(basis, integrand1, [vector], quadrature1)
mlhp.integrateOnSurface(basis, integrand2, [matrix, vector], quadrature2)
mlhp.integrateOnSurface(basis, integrand3, [matrix, vector], quadrature3)

### 5. Computing domain integral

In [21]:
E = mlhp.scalarField(D, youngsModulus)
nu = mlhp.scalarField(D, poissonsRatio)
rhs = mlhp.vectorField(D, [0.0, 0.0, 0.0]) #Body forces applied on the body. 

kinematics = mlhp.smallStrainKinematics(D)
constitutive = mlhp.isotropicElasticMaterial(E, nu)
integrand = mlhp.staticDomainIntegrand(kinematics, constitutive, rhs)

quadrature = mlhp.momentFittingQuadrature(domain, depth=polynomialDegree, epsilon=alphaFCM)

mlhp.integrateOnDomain(basis, integrand, [matrix, vector],
quadrature=quadrature)

### 6. Solving linear system

In [22]:
#P = mlhp.additiveSchwarzPreconditioner(matrix, basis, dirichlet[0])
P = mlhp.diagonalPreconditioner(matrix)

dofs, norms = mlhp.cg(matrix, vector, M=P, maxiter=10000, residualNorms=True)
dofs_vec=dofs#dofs original abspeichern 
#print(f"cond K after domain integral:{numpy.linalg.cond(matrix.todense())}")
#import matplotlib.pyplot as plt
#plt.loglog(norms)
#plt.show()

### 7. Postprocessing solution

In [23]:
# Ergebnis-Ordner anlegen
nelem_str = f"{nelements[0]}x{nelements[1]}x{nelements[2]}"
output_subdir = os.path.join(outputDir, f"mesh_{nelem_str}")
os.makedirs(output_subdir, exist_ok=True)

# Gradient und Feldprozessoren
gradient = mlhp.projectGradient(basis, dofs, quadrature)
processors = [
    mlhp.solutionProcessor(D, dofs, "Displacement"),
    mlhp.stressProcessor(gradient, kinematics, constitutive),
    mlhp.vonMisesProcessor(dofs, kinematics, constitutive, "VonMises1"),
#   mlhp.vonMisesProcessor(gradient, kinematics, constitutive, "VonMises2"),
#   mlhp.strainEnergyProcessor(gradient, kinematics, constitutive),
    mlhp.functionProcessor(domain)
]
print(dofs)
# STL-Schnitt und Surface Mesh
intersected, celldata = mlhp.intersectTriangulationWithMesh(grid, triangulation, kdtree)
surfmesh = mlhp.associatedTrianglesCellMesh(intersected, celldata)

# Ausgabe auf STL-Schnitt
writer0 = mlhp.PVtuOutput(filename=os.path.join(output_subdir, "stl_boundary"))
writer1 = mlhp.PVtuOutput(filename=os.path.join(output_subdir, "fcmmesh"))

mlhp.writeBasisOutput(basis, surfmesh, writer0, processors)
mlhp.writeBasisOutput(basis, writer=writer1, processors=processors)

# Weitere STL-Boundary-Meshes (z. B. für symmetrische Seiten)
surfmesh0 = mlhp.associatedTrianglesCellMesh(intersected0, celldata0)
surfmesh1 = mlhp.associatedTrianglesCellMesh(intersected1, celldata1)
surfmesh2 = mlhp.associatedTrianglesCellMesh(intersected2, celldata2)
surfmesh3 = mlhp.associatedTrianglesCellMesh(intersected3, celldata3)

surfwriter0 = mlhp.VtuOutput(filename=os.path.join(output_subdir, "boundary0"))
surfwriter1 = mlhp.VtuOutput(filename=os.path.join(output_subdir, "boundary1"))
surfwriter2 = mlhp.VtuOutput(filename=os.path.join(output_subdir, "boundary2"))
surfwriter3 = mlhp.VtuOutput(filename=os.path.join(output_subdir, "boundary3"))

mlhp.writeMeshOutput(grid, surfmesh0, surfwriter0, [])
mlhp.writeMeshOutput(grid, surfmesh1, surfwriter1, [])
mlhp.writeMeshOutput(grid, surfmesh2, surfwriter2, [])
mlhp.writeMeshOutput(grid, surfmesh3, surfwriter3, [])

#### E_eff mit Strain Energy = $1/2*u^T*K*u$

In [24]:
# Strain energy berechnen
Ku = matrix * dofs_vec
strainEnergy_dot = 0.5 * np.dot(dofs_vec, Ku)# in Joule (N·m)
print("Strain energy (new, 0.5 * u^T Ku) in Joule: %e" % strainEnergy_dot)

#effektives Elastizitätsmodul
volume = lengths[0] * lengths[1] * lengths[2]
E_eff = 2 * strainEnergy_dot / (strain**2 * scale)# Ergebnis in Pa
print(f"E_eff in Pa: {E_eff:.2e}")
print(f"E_eff in MPa: {E_eff / 1e6:.2f}")
print("E_solid in MPa: ", youngsModulus/1e6)#zum vergleich

Strain energy (new, 0.5 * u^T Ku) in Joule: 7.999996e+17
E_eff in Pa: 1.60e+20
E_eff in MPa: 159999912088017.28
E_solid in MPa:  10000.0


### 8. Save data in CSV-File

In [25]:
print("Postprocessing: Verschiebung an 10 Punkten (direkte Auswertung)", flush=True)

evaluator = mlhp.vectorEvaluator(basis, dofs)
probe_results = []

for ID, pt in selected_pts:
    u_vec = evaluator(pt)
    mag = np.linalg.norm(u_vec)
    probe_results.append((ID, *pt, mag))

# CSV-Schreiben
output_path = outputDir + "/convergence.csv"
with open(output_path, "a") as f:
    if os.stat(output_path).st_size == 0:
        f.write("DOFs,StrainEnergyDot,TimeSeconds,ElementsPerDim")
        for i in range(10):
            f.write(f",X{i},Y{i},Z{i},U{i}")
        f.write("\n")

    elapsed = datetime.datetime.now() - starttime
    total_seconds = elapsed.total_seconds()
    f.write(f"{dofs.size},{strainEnergy_dot:.8e},{total_seconds:.2f},{nelements[0]}")

    for row in probe_results:
        _, x, y, z, u = row
        f.write(f",{x:.6f},{y:.6f},{z:.6f},{u:.6e}")
    f.write("\n")


Postprocessing: Verschiebung an 10 Punkten (direkte Auswertung)
